# Day 26 — Multi-LoRA Serving

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day26-multi-lora-serving.ipynb)

**Big idea:** one frozen base model, many tiny low-rank adapters routed per request.
CPU runs cells 1–7 (~30 min). Cell 8 is the optional T4 vLLM demo.

In [ ]:
# Colab: install deps (only cell that installs)
%pip install --quiet torch
print('deps ok')
# Expected: deps ok

## 1. Setup
Pure PyTorch. No model downloads — the LoRA math is the lesson.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
# Expected: device: cpu  (or cuda on a T4 runtime)

## 2. LoRALinear: frozen W, trainable B and A
`W' = W + B A`, with B of shape (d, r) and A of shape (r, d). Forward: `xW + xBA`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, d=512, r=8):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d, d) * 0.02, requires_grad=False)
        self.B = nn.Parameter(torch.zeros(d, r))
        self.A = nn.Parameter(torch.randn(r, d) * 0.01)

    def forward(self, x):
        return x @ self.W + (x @ self.A.T) @ self.B.T

layer = LoRALinear(d=512, r=8)
frozen = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
print(f"frozen: {frozen}  lora trainable: {trainable}  ({frozen/trainable:.1f}x fewer)")
# Expected: frozen: 262144  lora trainable: 8192  (32.0x fewer)

## 3. The merge identity
Merging computes `W + B A` once and serves a plain model. The two paths must agree to float tolerance — this is why serving can keep adapters separate.

In [ ]:
x = torch.randn(4, 512)
with torch.no_grad():
    via_adapter = layer(x)
    merged = x @ (layer.W + layer.B @ layer.A)
diff = (via_adapter - merged).abs().max().item()
print(f"merge max abs diff: {diff:.2e}  OK")
assert diff < 1e-5
# Expected: merge max abs diff: < 1e-5  OK

## 4. Adapter memory calculator (Llama-3.1-8B)
7 projections per layer, 32 layers, d=4096. Rank scales linearly.

In [ ]:
def adapter_stats(d=4096, r=16, matrices=7, layers=32, base_params=8.03e9, dtype_bytes=2):
    per_matrix = 2 * d * r
    params = per_matrix * matrices * layers
    mb = params * dtype_bytes / 1e6
    flop_overhead = (2 * params) / (2 * base_params) * 100
    return params, mb, flop_overhead

for r in (8, 16, 32):
    p, mb, ov = adapter_stats(r=r)
    print(f"r={r:2d}: {p/1e6:6.2f}M params  {mb:6.1f} MB/adapter  overhead {ov:.2f}%")
p, mb, ov = adapter_stats()
for n in (10, 50, 100):
    print(f"{n:4d} adapters: {n*mb/1000:5.2f} GB  (+ 16 GB base fp16)")
# Expected: r=16: 29.36M params  58.7 MB/adapter  overhead 0.37%
#           100 adapters: 5.87 GB  (+ 16 GB base fp16)

## 5. Train two tiny adapters (toy style tasks)
Same frozen base, two different low-rank deltas. 50 CPU steps each. Adapters 'style_a' and 'style_b' learn different target rotations of the input.

In [ ]:
def train_adapter(seed, steps=50):
    torch.manual_seed(seed)
    layer = LoRALinear(d=512, r=8)
    R = torch.linalg.qr(torch.randn(512, 512))[0]  # random orthogonal target
    X = torch.randn(64, 512)
    Y = X @ R.T
    opt = torch.optim.Adam([layer.B, layer.A], lr=0.05)
    for i in range(steps):
        opt.zero_grad()
        loss = ((layer(X) - Y) ** 2).mean()
        loss.backward()
        opt.step()
    return layer, loss.item()

la, final_loss_a = train_adapter(seed=1)
lb, final_loss_b = train_adapter(seed=2)
x1 = torch.randn(1, 512)
with torch.no_grad():
    outs_differ = not torch.allclose(la(x1), lb(x1), atol=1e-3)
print(f"final losses: a={final_loss_a:.4f} b={final_loss_b:.4f}")
print(f"same input, different adapters -> different outputs: {outs_differ}")
print(f"adapter params each: {sum(p.numel() for p in (la.B, la.A))}")
# Expected: outputs differ: True;  each adapter is 8192 params

## 6. Routing simulation: heterogeneous batch
vLLM's Punica kernel does this per segment; we simulate the routing logic: each sequence carries an adapter id, the base GEMM is shared, the delta is per-sequence.

In [ ]:
adapters = {"base": None, "style_a": la, "style_b": lb}
requests = [("req1", "style_a"), ("req2", "style_b"), ("req3", "base"), ("req4", "style_a")]
batch = torch.randn(len(requests), 512)
with torch.no_grad():
    base_out = batch @ la.W  # one shared base GEMM for the whole batch
    for i, (rid, name) in enumerate(requests):
        ad = adapters[name]
        out = base_out[i] if ad is None else base_out[i] + (batch[i] @ ad.A.T) @ ad.B.T
        print(f"{rid}: adapter={name:8s} out[:3]={[round(v, 3) for v in out[:3].tolist()]}")
print("base GEMM computed once; deltas applied per sequence")
# Expected: 4 lines; req1 and req4 (same adapter, diff inputs) track style_a

## 7. Overhead recap
The numbers from cell 4, in words: ~0.4% FLOPs, ~59 MB per adapter. Rank is a quality knob, not a speed knob.

In [ ]:
p, mb, ov = adapter_stats()
print(f"one rank-16 adapter: {mb:.1f} MB, {ov:.2f}% of base FLOPs/token")
print(f"100 adapters: {100*mb/1000:.2f} GB vs 100 full fine-tunes: {100*16:.0f} GB")
print("verdict: 100 models, 1 GPU")
# Expected: one rank-16 adapter: 58.7 MB, 0.37% of base FLOPs/token

## 8. (Optional, T4) Serve base + 2 adapters in vLLM
Run in a T4 runtime. Downloads a small base model; adapters below are placeholders for your PEFT-saved dirs from a real fine-tune. Skip on CPU.

In [ ]:
import sys
if not torch.cuda.is_available():
    print("no GPU: skip the vLLM serving demo (cells 1-7 are the core lab)")
else:
    print("Serving sketch (run in a terminal, not this cell):")
    print("  pip install -q vllm peft")
    print("  vllm serve Qwen/Qwen2.5-0.5B-Instruct --enable-lora --max-loras 2 --lora-modules style_a=/path/a style_b=/path/b")
    payload = {"model": "style_a", "messages": [{"role": "user", "content": "Say ahoy"}], "max_tokens": 40}
    print("  POST /v1/chat/completions with JSON body:", payload)
    print("route per request via the model field; record TPOT for base vs each adapter")
# Expected (CPU): no GPU: skip the vLLM serving demo

## Checkpoints (answer before reading)
1. One projection, d=4096, rank 16: how many params vs full? → 131,072 vs 16.78M (128× fewer).
2. Why can 100 adapters share one GPU with one base? → Base shared; ~59 MB deltas; batched low-rank kernels.
3. Per-token FLOP overhead of the LoRA path on 8B? → ~0.4%.

**Write-up:** fill the packet's 'What to measure' table and write the '100 models, 1 GPU' paragraph citing your cell-4 numbers.